In [ ]:
from pathlib import Path

import os
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/numba_cache")

import json
import re
import jieba
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from scipy.cluster.hierarchy import linkage
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from tqdm.auto import tqdm

# CSV_PATH = Path("output/0623/xizhan_negative_texts.csv")
MODEL_PATH = Path("embModel/Qwen3-Embedding-0.6B")
KEYWORD_FILTER_CONFIG_PATH = Path("config/station_keyword_filter.json")
STOPWORD_PATHS = [Path("stopword.txt"), Path("stopinthis.txt")]
RESULT_PATH = Path("output/0623/exp/bertopic_kmeans_results.csv")
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)


BATCH_SIZE = 16
MIN_TEXT_LENGTH = 4
N_WORDS = 10
N_TOPICS = 10
K_SWEEP_RANGE = range(5, 16)


### 0. 数据读取与字段规范化

这一组单元负责读取新版 `llmOutput` 数据，并把后续分析需要的字段整理成统一口径。


In [ ]:
# 优先读取新版 llmOutput；若目录不存在，再回退到旧的中间产物目录。
LLM_OUTPUT_CANDIDATES = [Path("llmOutput"), Path("llmPart/outputs"), Path("analysis_json")]
analysis_paths = []
LLM_OUTPUT_DIR = None

for candidate in LLM_OUTPUT_CANDIDATES:
    candidate_paths = sorted(candidate.glob("*/*_analysis.json"))
    if candidate_paths:
        LLM_OUTPUT_DIR = candidate
        analysis_paths = candidate_paths
        break

if not analysis_paths:
    searched = "、".join(str(path.resolve()) for path in LLM_OUTPUT_CANDIDATES)
    raise FileNotFoundError(f"没有找到 *_analysis.json 文件，已搜索：{searched}")

records = []
for analysis_path in analysis_paths:
    with analysis_path.open("r", encoding="utf-8") as f:
        station_records = json.load(f)
    if not isinstance(station_records, list):
        raise ValueError(f"{analysis_path} 的顶层结构应为 list")

    station = analysis_path.parent.name
    for record in station_records:
        if isinstance(record, dict):
            records.append({**record, "station": station})

raw = pd.DataFrame(records)
print(f"读取分析目录：{LLM_OUTPUT_DIR.resolve()}；分析文件：{len(analysis_paths):,} 个；原始记录：{len(raw):,} 条")

# 统一情感字段：优先使用 llm_sent；旧版数据回退到 llm_sentiment。
if "llm_sent" in raw.columns:
    SENTIMENT_SOURCE_COLUMN = "llm_sent"
elif "llm_sentiment" in raw.columns:
    SENTIMENT_SOURCE_COLUMN = "llm_sentiment"
else:
    raise KeyError("数据中缺少 llm_sent 或 llm_sentiment，无法进行情感筛选")

raw["llm_sent"] = raw[SENTIMENT_SOURCE_COLUMN]
raw["llm_sentiment"] = raw["llm_sent"]
print(f"情感筛选字段：{SENTIMENT_SOURCE_COLUMN}")


In [ ]:
# 新版 llmOutput 同时保留原帖 content 和 LLM description。
# 聚类/主题发现优先使用结构化的 LLM description；没有 description 时回退到原帖 content。
description_text = raw["description"].fillna("").astype(str).str.strip() if "description" in raw.columns else pd.Series("", index=raw.index)
post_text = raw["content"].fillna("").astype(str).str.strip() if "content" in raw.columns else pd.Series("", index=raw.index)

raw["source_content"] = post_text
raw["analysis_text"] = description_text.where(description_text.ne(""), post_text)
raw.head()


### 0.1 负向舆情筛选与基础配置

本 notebook 的分析样本由 `llm_sent`（不存在时回退为 `llm_sentiment`）筛选，仅保留负向文本。


In [ ]:
# 业务类别仅用于后续描述、严重性与交叉对照；实验样本由 llm_sent 的负向标签决定。
ENGAGEMENT_COLUMNS = ["liked_count", "collected_count", "comment_count", "share_count"]
BUSINESS_CATEGORY_V2_TARGETS = [
    "导航导视指引不足，导致通行效率低",
    "智慧接驳衔接不畅，最后一公里体验亟待提升",
    "室内外环境问题",
    "站区服务和商业配套",
    "城管管理，周边治理",
]
BUSINESS_CATEGORY_V2_LABELS = {category: category for category in BUSINESS_CATEGORY_V2_TARGETS}


In [ ]:
# 业务严重性规则：先按 5 类业务类别给基础风险，再用风险关键词补充具体场景信号。
SEVERITY_CATEGORY_BASE = {
    "导航导视指引不足，导致通行效率低": 86,
    "智慧接驳衔接不畅，最后一公里体验亟待提升": 82,
    "室内外环境问题": 72,
    "站区服务和商业配套": 60,
    "城管管理，周边治理": 78,
}

SEVERITY_KEYWORD_RULES = {
    "安全/应急": (18, ["漏电", "电击", "摔", "事故", "冲撞", "危险", "隐患", "报警", "乘警", "失控"]),
    "严重受阻": (14, ["严重", "崩溃", "瘫痪", "大面积", "堵死", "排长队", "巨堵", "拥挤", "挤爆"]),
    "误车/延误": (13, ["赶不上", "误车", "停运", "晚点", "取消", "改签", "延误", "滞留", "来不及"]),
    "服务失灵": (10, ["没人管", "无人处理", "态度差", "故障", "坏了", "失灵", "投诉", "摆烂"]),
    "体验伤害": (8, ["难受", "恶心", "异味", "脏", "冷", "热", "疼", "头晕", "受不了"]),
}


### 0.2 通用辅助函数

这些函数负责互动量解析、百分位转换、风险评分和 Pareto 分层。


In [ ]:
def parse_count(value) -> float:
    """把 '1.2万'、'300' 等互动量字段统一转成数字。"""
    if value is None or pd.isna(value):
        return 0.0

    text = str(value).replace(",", "").strip()
    if not text:
        return 0.0

    try:
        if text.endswith("万"):
            return float(text[:-1]) * 10000
        if text.endswith("亿"):
            return float(text[:-1]) * 100000000
        return float(text)
    except ValueError:
        return 0.0


In [ ]:
# 预计算语义相似矩阵，并定义按分组构图与 PageRank 的通用函数。
SIMILARITY_TOP_K = 8
SIMILARITY_THRESHOLD = 0.42
def percentile_score(series: pd.Series) -> pd.Series:
    """把连续值转成 0-100 的百分位分，便于不同量纲比较。"""
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.nunique(dropna=False) <= 1:
        return pd.Series(50.0, index=series.index)
    return series.rank(method="average", pct=True) * 100
def build_group_semantic_graph(
    group_values,
    similarity_matrix_input=None,
    top_k: int = SIMILARITY_TOP_K,
    threshold: float = SIMILARITY_THRESHOLD,
) -> nx.Graph:
    """在同一分组内部连接语义最相似的帖子。"""
    groups = pd.Series(group_values).reset_index(drop=True)
    matrix = similarity_matrix if similarity_matrix_input is None else np.asarray(similarity_matrix_input)
    graph = nx.Graph()
    graph.add_nodes_from(range(len(groups)))

    for _, indices in groups.groupby(groups).groups.items():
        indices = list(indices)
        if len(indices) <= 1:
            continue

        group_similarity = matrix[np.ix_(indices, indices)]
        for local_i, source_idx in enumerate(indices):
            neighbor_order = group_similarity[local_i].argsort()[::-1][:top_k]
            for local_j in neighbor_order:
                target_idx = indices[local_j]
                sim = float(group_similarity[local_i, local_j])
                if source_idx == target_idx or sim < threshold:
                    continue
                graph.add_edge(source_idx, target_idx, weight=sim)

    if graph.number_of_edges() == 0 and len(groups) > 1:
        graph.add_edges_from((idx, idx + 1, {"weight": 1.0}) for idx in range(len(groups) - 1))
    return graph


def personalized_pagerank(graph: nx.Graph, engagement: pd.Series) -> pd.Series:
    """用互动总量作为 PageRank 个性化启动概率，不给单个互动字段设置人工权重。"""
    personalization_raw = np.log1p(engagement.astype(float).reset_index(drop=True))
    if float(personalization_raw.sum()) == 0:
        personalization = {idx: 1 / len(engagement) for idx in range(len(engagement))}
    else:
        personalization = (personalization_raw / personalization_raw.sum()).to_dict()

    scores = nx.pagerank(
        graph,
        alpha=0.85,
        personalization=personalization,
        weight="weight",
        max_iter=200,
        tol=1e-8,
    )
    return pd.Series(scores).sort_index()


In [ ]:
def normalize_llm_sentiment(value) -> str:
    """把 LLM 情感标签统一为 negative / neutral / positive / unknown。"""
    label = str(value or "").strip().lower()
    if label in {"负向", "负面", "negative", "neg"} or "负" in label:
        return "negative"
    if label in {"正向", "正面", "positive", "pos"} or "正" in label:
        return "positive"
    if label in {"中立", "中性", "neutral", "neu"} or "中" in label:
        return "neutral"
    return "unknown"


def negative_pressure(row: pd.Series) -> float:
    """基于 LLM 情感标签估计负向压力。"""
    llm_sentiment_group = normalize_llm_sentiment(row.get("llm_sentiment", ""))
    score = float(row.get("sentiment_score", 0) or 0)
    score = max(0.0, min(1.0, score))

    if llm_sentiment_group == "negative":
        return 70 + 30 * score
    if llm_sentiment_group == "positive":
        return 15 + 45 * (1 - score)
    return 50


def severity_score(row: pd.Series) -> float:
    """根据 5 类业务类别和风险关键词估计治理严重性。"""
    text = f"{row.get('title', '')} {row.get('content', '')} {row.get('source_content', '')}"
    score = SEVERITY_CATEGORY_BASE.get(row.get("business_category_v2", ""), 55)
    for _, (bonus, keywords) in SEVERITY_KEYWORD_RULES.items():
        if any(keyword in text for keyword in keywords):
            score += bonus
    return min(float(score), 100.0)


def evidence_score(row: pd.Series) -> float:
    """估计内容作为治理证据的完整程度。"""
    text = f"{row.get('title', '')} {row.get('content', '')} {row.get('source_content', '')}".strip()
    image_score = min(parse_count(row.get("num_images", 0)), 6) / 6 * 35
    length_score = min(len(text), 280) / 280 * 45
    category_score = 20 if row.get("business_category_v2") in BUSINESS_CATEGORY_V2_TARGETS else 8
    return min(image_score + length_score + category_score, 100.0)


def clean_preview(text, max_chars: int = 90) -> str:
    """把原文压成一行短预览，避免标题候选表过长。"""
    value = " ".join(str(text).split())
    if len(value) <= max_chars:
        return value
    return value[:max_chars].rstrip() + "..."

In [ ]:
def pareto_front_rank(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    """多目标非支配分层：层级越小，说明越值得优先复核。"""
    values = frame[columns].apply(pd.to_numeric, errors="coerce").fillna(0).to_numpy(dtype=float)
    remaining = set(range(len(frame)))
    ranks = np.zeros(len(frame), dtype=int)
    layer = 1

    while remaining:
        front = []
        for i in remaining:
            dominated = False
            for j in remaining:
                if i == j:
                    continue
                if np.all(values[j] >= values[i]) and np.any(values[j] > values[i]):
                    dominated = True
                    break
            if not dominated:
                front.append(i)

        for i in front:
            ranks[i] = layer
        remaining.difference_update(front)
        layer += 1

    return pd.Series(ranks, index=frame.index)


## 负向舆情实验分析


### 负向舆情：1. 数据筛选

只保留 `llm_sent` 判定为负向的文本；`business_category_v2` 仅作为结果解释字段，不参与样本筛选。


In [ ]:
# 实验样本：只保留 llm_sent（兼容 llm_sentiment）的负向文本。
# analysis_text 用于 KMeans 语义聚类，source_content 保留原帖正文以便追溯。
base_columns = [
    "station", "note_id", "title", "category", "business_category_v2", "category_group",
    "analysis_text", "source_content", "sentiment_label", "sentiment_score", "llm_sent", "llm_sentiment",
    "num_images", "upload_time",
]
source_columns = [column for column in base_columns + ENGAGEMENT_COLUMNS if column in raw.columns]
source_df = raw[source_columns].copy()
source_df["llm_sentiment_group"] = source_df["llm_sent"].map(normalize_llm_sentiment)
source_df = source_df.loc[source_df["llm_sentiment_group"].eq("negative")].copy()

clean_content = source_df["analysis_text"].fillna("").astype(str).str.strip()
keep = clean_content.ne("") & clean_content.ne("无标题") & clean_content.str.len().ge(MIN_TEXT_LENGTH)

result_df = source_df.loc[keep].rename(columns={"note_id": "id", "analysis_text": "content"}).reset_index(drop=True)
documents = clean_content.loc[keep].tolist()
result_df["content"] = documents

# 负向实验以整个负向语料为分析范围；业务字段只保留作后续解释和交叉对照。
result_df["llm_category"] = "负向舆情"
result_df["llm_category_label"] = "负向舆情"

if len(documents) < N_TOPICS:
    raise ValueError(f"可聚类负向文本仅有 {len(documents)} 条，少于 N_TOPICS={N_TOPICS}")

print(f"llm_sent 负向文本：{len(documents):,} 条")
display(result_df["llm_sentiment_group"].value_counts().rename_axis("llm_sentiment_group").reset_index(name="文本数"))
result_df.head()


In [ ]:
# 解析互动量；LLM 分类和 KMeans 主题分析都会复用这些基础互动字段。
for column in ENGAGEMENT_COLUMNS:
    if column not in result_df.columns:
        result_df[column] = 0
    result_df[f"{column}_num"] = result_df[column].apply(parse_count)

result_df["互动总量"] = result_df[[f"{column}_num" for column in ENGAGEMENT_COLUMNS]].sum(axis=1)
result_df["互动强度"] = percentile_score(np.log1p(result_df["互动总量"])).round(2)

result_df[["id", "station", "llm_category_label", "互动总量", "互动强度"]].head()


### 负向舆情：2. 计算 PageRank

在全部负向文本上建立语义相似网络并计算 PageRank；embedding 将被后续 KMeans 实验复用。


In [ ]:
if not (MODEL_PATH / "config.json").is_file():
    raise FileNotFoundError(f"找不到 Hugging Face 模型：{MODEL_PATH.resolve()}")

try:
    import torch
    EMBEDDING_DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
except Exception:
    EMBEDDING_DEVICE = "cpu"

print(f"Embedding device: {EMBEDDING_DEVICE}")
embedding_model = SentenceTransformer(
    str(MODEL_PATH),
    device=EMBEDDING_DEVICE,
    trust_remote_code=True,
    local_files_only=True,
)
embedding_batches = []
for start in tqdm(range(0, len(documents), BATCH_SIZE), desc="Encoding embeddings", unit="batch"):
    batch_documents = documents[start:start + BATCH_SIZE]
    batch_embeddings = embedding_model.encode(
        batch_documents,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    embedding_batches.append(batch_embeddings)

embeddings = np.vstack(embedding_batches) if embedding_batches else np.empty((0, 0))
print(f"Embedding shape: {embeddings.shape}")


In [ ]:
# 预计算语义相似矩阵，并定义按分组构图与 PageRank 的通用函数。
SIMILARITY_TOP_K = 8
SIMILARITY_THRESHOLD = 0.42

# embeddings 已经 normalize，因此点积就是余弦相似度。
similarity_matrix = np.asarray(embeddings) @ np.asarray(embeddings).T
np.fill_diagonal(similarity_matrix, 0)

In [ ]:
# 负向实验 PageRank：在全部负向文本的语义网络内连边。
llm_graph = build_group_semantic_graph(pd.Series("负向舆情", index=result_df.index))
llm_pagerank = personalized_pagerank(llm_graph, result_df["互动总量"])

result_df["llm_pagerank"] = llm_pagerank.to_numpy()
result_df["llm_impact_score"] = (result_df["llm_pagerank"] * 10000).round(4)
result_df["llm_network_centrality"] = percentile_score(result_df["llm_pagerank"]).round(2)
result_df["llm_category_recurrence"] = percentile_score(result_df["llm_category"].map(result_df["llm_category"].value_counts())).round(2)

print(f"LLM 分类语义网络：{llm_graph.number_of_nodes():,} 个节点，{llm_graph.number_of_edges():,} 条边")
result_df[["id", "llm_category_label", "llm_impact_score", "llm_network_centrality", "互动总量", "llm_category_recurrence"]].head()


### 负向舆情：3. 影响力与帖子可视化


In [ ]:
# 负向舆情的风险指标在本章节计算，不参与后面的 KMeans 选 K。
result_df["负向压力"] = result_df.apply(negative_pressure, axis=1).round(2)
result_df["治理严重性"] = result_df.apply(severity_score, axis=1).round(2)
result_df["矛盾风险度"] = np.maximum(result_df["负向压力"], result_df["治理严重性"]).round(2)
result_df["证据完整度"] = result_df.apply(evidence_score, axis=1).round(2)

total_docs = len(result_df)
total_engagement = result_df["互动总量"].sum()
total_llm_impact = result_df["llm_impact_score"].sum()

llm_category_summary = (
    result_df.groupby(["llm_category", "llm_category_label"], as_index=False)
    .agg(
        文本数=("id", "count"),
        点赞数=("liked_count_num", "sum"),
        收藏数=("collected_count_num", "sum"),
        评论数=("comment_count_num", "sum"),
        转发数=("share_count_num", "sum"),
        互动总量=("互动总量", "sum"),
        综合影响分=("llm_impact_score", "sum"),
        平均网络中心性=("llm_network_centrality", "mean"),
        平均互动强度=("互动强度", "mean"),
        矛盾风险度=("矛盾风险度", "mean"),
        负向压力=("负向压力", "mean"),
        治理严重性=("治理严重性", "mean"),
        证据完整度=("证据完整度", "mean"),
    )
    .reset_index(drop=True)
)
llm_category_summary["文本占比"] = llm_category_summary["文本数"] / total_docs
llm_category_summary["互动占比"] = llm_category_summary["互动总量"] / total_engagement if total_engagement else 0
llm_category_summary["影响占比"] = llm_category_summary["综合影响分"] / total_llm_impact if total_llm_impact else 0
llm_category_summary["业务优先级层"] = pareto_front_rank(llm_category_summary, ["综合影响分", "矛盾风险度", "文本数"])
score_columns = ["综合影响分", "平均网络中心性", "平均互动强度", "矛盾风险度", "负向压力", "治理严重性", "证据完整度"]
llm_category_summary[score_columns] = llm_category_summary[score_columns].round(2)
llm_category_summary = llm_category_summary.sort_values(["业务优先级层", "综合影响分"], ascending=[True, False]).reset_index(drop=True)

llm_category_summary.to_csv(RESULT_PATH.with_name("negative_sentiment_summary.csv"), index=False, encoding="utf-8-sig")
display(llm_category_summary[["业务优先级层", "llm_category_label", "文本数", "文本占比", "综合影响分", "影响占比", "互动总量", "互动占比", "平均网络中心性", "矛盾风险度", "负向压力", "治理严重性", "证据完整度"]])
print(f"LLM 五类业务汇总已保存到：{RESULT_PATH.with_name('negative_sentiment_summary.csv').resolve()}")


In [ ]:
# LLM 分类影响力占比图。
llm_category_chart_data = llm_category_summary.melt(
    id_vars="llm_category_label",
    value_vars=["文本占比", "影响占比"],
    var_name="指标",
    value_name="占比",
)

# # LLM 分类数量占比图。
# llm_category_chart_data = llm_category_summary.melt(
#     id_vars="llm_category_label",
#     value_vars=["文本数", "综合影响分"],
#     var_name="指标",
#     value_name="占比",
# )
fig_llm_category = px.bar(
    llm_category_chart_data,
    x="占比",
    y="llm_category_label",
    color="指标",
    orientation="h",
    barmode="group",
    text=llm_category_chart_data["占比"].map(lambda value: f"{value:.1%}"),
    title="负向舆情：文本占比 vs 影响占比",
    labels={"llm_category_label": "负向舆情范围", "占比": "占比"},
)

fig_llm_category.update_traces(textposition="outside")
fig_llm_category.update_layout(height=max(420, 85 * len(llm_category_summary)), margin={"l": 180, "r": 40, "t": 60, "b": 40})
fig_llm_category.update_xaxes(tickformat=".0%")
fig_llm_category.show()

llm_category_chart_path = RESULT_PATH.with_name("negative_sentiment_distribution.html")
fig_llm_category.write_html(llm_category_chart_path, include_plotlyjs=True)
print(f"LLM 五类业务占比图已保存到：{llm_category_chart_path.resolve()}")


In [ ]:
# LLM 分类帖子级影响力图：每个点是一条帖子。
llm_post_df = result_df.copy()
llm_post_df["文本预览"] = llm_post_df["source_content"].fillna(llm_post_df["content"]).map(lambda text: clean_preview(text, max_chars=90))
llm_post_df["风险气泡"] = llm_post_df["矛盾风险度"].clip(lower=1)

fig_llm_posts = px.scatter(
    llm_post_df,
    x="llm_impact_score",
    y="互动总量",
    color="llm_category_label",
    size="风险气泡",
    hover_data={
        "station": True,
        "llm_category_label": True,
        "llm_impact_score": ":.2f",
        "llm_network_centrality": ":.1f",
        "矛盾风险度": ":.1f",
        "文本预览": True,
        "风险气泡": False,
    },
    title="LLM 分类：帖子影响力与互动量",
    labels={"llm_impact_score": "LLM 类内 PageRank 影响分", "llm_category_label": "负向舆情范围"},
)
fig_llm_posts.update_traces(marker={"opacity": 0.75, "line": {"width": 0.4, "color": "white"}})
fig_llm_posts.update_layout(height=620, margin={"l": 60, "r": 260, "t": 70, "b": 50}, legend_title_text="负向舆情范围")
fig_llm_posts.show()

llm_posts_path = RESULT_PATH.with_name("negative_sentiment_post_influence.html")
fig_llm_posts.write_html(llm_posts_path, include_plotlyjs=True)

llm_top_posts = llm_post_df.sort_values("llm_impact_score", ascending=False).head(30)
llm_top_posts[["id", "station", "llm_category_label", "llm_impact_score", "llm_network_centrality", "互动总量", "矛盾风险度", "文本预览"]].to_csv(
    RESULT_PATH.with_name("negative_top_influence_posts.csv"), index=False, encoding="utf-8-sig"
)
display(llm_top_posts[["id", "station", "llm_category_label", "llm_impact_score", "llm_network_centrality", "互动总量", "矛盾风险度", "文本预览"]])
print(f"LLM 帖子影响力图已保存到：{llm_posts_path.resolve()}")


## Kmeans 实验

### KMeans：1. LLM 情感筛选、复用 embedding 并选择 K

KMeans 部分先按 LLM 情感标签 `llm_sentiment` 生成专用建模数据：可选择“中立+负向”或“仅负向”。筛选后得到 `kmeans_df`、`kmeans_documents` 和 `kmeans_embeddings`，后续 K 值选择、聚类、关键词和影响力计算都只使用这套数据。


In [ ]:
# 实验 notebook 的 result_df 已经由 llm_sent 筛选为全部负向文本。
# 直接复用前文 embedding，避免再次编码同一批数据。
KMEANS_DATA_SOURCE = "all_negative"
KMEANS_SENTIMENT_SCOPE = "negative"

kmeans_df = result_df.copy().reset_index(drop=True)
kmeans_documents = kmeans_df["content"].fillna("").astype(str).tolist()
kmeans_embeddings = embeddings.copy()

if len(kmeans_documents) < 2:
    raise ValueError(f"llm_sent 筛选后仅有 {len(kmeans_documents)} 条负向文本，无法聚类")

print(f"KMeans 数据源：llm_sent 负向数据（{len(kmeans_documents):,} 条）")
display(
    kmeans_df["llm_sentiment_group"]
    .value_counts()
    .rename_axis("kmeans_sentiment_group")
    .reset_index(name="KMeans建模文本数")
)


In [ ]:
def small_topic_count(labels, min_share: float = 0.03) -> int:
    """统计占比过小的主题数量，过多说明 K 可能偏大。"""
    shares = pd.Series(labels).value_counts(normalize=True)
    return int((shares < min_share).sum())


In [ ]:
def evaluate_kmeans_k(k: int, embeddings_array) -> dict:
    """对单个 K 训练 KMeans，并返回无监督聚类结构指标。"""
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(embeddings_array)
    shares = pd.Series(labels).value_counts(normalize=True)
    return {
        "k": k,
        "silhouette_cosine": silhouette_score(embeddings_array, labels, metric="cosine"),
        "davies_bouldin": davies_bouldin_score(embeddings_array, labels),
        "calinski_harabasz": calinski_harabasz_score(embeddings_array, labels),
        "small_topics": small_topic_count(labels),
        "min_topic_share": float(shares.min()),
    }


In [ ]:
candidate_ks = [k for k in K_SWEEP_RANGE if 2 <= k < len(kmeans_documents)]
k_metrics = pd.DataFrame(
    evaluate_kmeans_k(k, kmeans_embeddings)
    for k in tqdm(candidate_ks, desc="KMeans K sweep", unit="k")
)

# 统一转成“越大越好”的百分位分，再合成一个辅助选择分。
k_metrics["silhouette_rank"] = percentile_score(k_metrics["silhouette_cosine"])
k_metrics["dbi_rank"] = percentile_score(-k_metrics["davies_bouldin"])
k_metrics["ch_rank"] = percentile_score(k_metrics["calinski_harabasz"])
k_metrics["min_share_rank"] = percentile_score(k_metrics["min_topic_share"])
k_metrics["small_topic_penalty"] = k_metrics["small_topics"] / max(k_metrics["k"].max(), 1) * 100


In [ ]:
k_metrics["score"] = (
    0.40 * k_metrics["silhouette_rank"]
    + 0.25 * k_metrics["dbi_rank"]
    + 0.20 * k_metrics["ch_rank"]
    + 0.15 * k_metrics["min_share_rank"]
    - 0.15 * k_metrics["small_topic_penalty"]
).round(4)

k_metrics = k_metrics.sort_values("score", ascending=False).reset_index(drop=True)
N_TOPICS = int(k_metrics.loc[0, "k"])
print(f"推荐 K = {N_TOPICS}")
display(k_metrics[["k", "score", "silhouette_cosine", "davies_bouldin", "calinski_harabasz", "small_topics", "min_topic_share"]])


In [ ]:
K_SELECTION_PATH = RESULT_PATH.with_name("kmeans_k_selection_metrics.csv")
K_SELECTION_CHART_PATH = RESULT_PATH.with_name("kmeans_k_selection_score.html")
k_metrics.to_csv(K_SELECTION_PATH, index=False, encoding="utf-8-sig")

fig_k = px.line(
    k_metrics.sort_values("k"),
    x="k",
    y="score",
    markers=True,
    title="KMeans 候选 K 综合评分",
    labels={"k": "K", "score": "综合评分"},
)
fig_k.write_html(K_SELECTION_CHART_PATH, include_plotlyjs=True)

try:
    fig_k.show()
except Exception as exc:
    print(f"当前环境无法直接显示 K 选择图，已保存 HTML：{K_SELECTION_CHART_PATH.resolve()}；原因：{exc}")

print(f"K 值选择指标已保存到：{K_SELECTION_PATH.resolve()}")


### KMeans：2. 中文分词、关键词过滤与主题建模

这一组单元负责准备中文 tokenizer、运行 KMeans-BERTopic，并用原始中文文本生成可读主题关键词。


In [ ]:
# 加载停用词和关键词过滤配置。
missing_files = [str(path) for path in STOPWORD_PATHS if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"找不到停用词文件：{', '.join(missing_files)}")

stop_words = {"无标题"}
for path in STOPWORD_PATHS:
    stop_words.update(word.strip() for word in path.read_text(encoding="utf-8").splitlines() if word.strip())

if not KEYWORD_FILTER_CONFIG_PATH.is_file():
    raise FileNotFoundError(f"找不到关键词过滤配置：{KEYWORD_FILTER_CONFIG_PATH.resolve()}")

with KEYWORD_FILTER_CONFIG_PATH.open("r", encoding="utf-8") as f:
    keyword_filter_config = json.load(f)

keyword_exclude_words = {word.strip() for word in keyword_filter_config.get("exclude_words", []) if str(word).strip()}
keyword_prefer_words = {word.strip() for word in keyword_filter_config.get("prefer_words", []) if str(word).strip()}
stop_words.update(keyword_exclude_words)

print(f"关键词过滤：排除 {len(keyword_exclude_words):,} 个词，优先 {len(keyword_prefer_words):,} 个词。")


In [ ]:
# 中文分词函数：只服务于“可读关键词”抽取；Qwen embedding 不直接产生词。
CHINESE_WORD = re.compile(r"^[\u4e00-\u9fff]{2,}$")


def tokenize_zh(text: str) -> list[str]:
    return [
        word.strip()
        for word in jieba.lcut(text)
        if len(word.strip()) > 1
        and not word.strip().isdigit()
        and word.strip() not in stop_words
    ]


def tokenize_chinese_only(text: str) -> list[str]:
    return [
        word.strip()
        for word in jieba.lcut(text)
        if CHINESE_WORD.fullmatch(word.strip()) and word.strip() not in stop_words
    ]


In [ ]:
# 构建 KMeans-BERTopic 模型。
# min_df 必须为 1：BERTopic 是按每个主题的汇总文本建立词表。
vectorizer_model = CountVectorizer(
    tokenizer=tokenize_zh,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 1),
    min_df=1,
)

cluster_model = KMeans(n_clusters=N_TOPICS, random_state=42, n_init=10)
topic_model = BERTopic(
    embedding_model=None,
    hdbscan_model=cluster_model,  # BERTopic 用这个参数接收任意聚类器
    vectorizer_model=vectorizer_model,
    top_n_words=50,  # 保留足够候选词，展示时再过滤英文/缩写噪声
    low_memory=True,
)

topics, _ = topic_model.fit_transform(kmeans_documents, kmeans_embeddings)
kmeans_df["topic"] = topics
print(f"已完成 {N_TOPICS} 个 KMeans 主题。")


In [ ]:
# 主题标签单独从原始中文文本计算，避免 BERTopic 内部词权被 tm/bro 等高频噪声劫持。
topic_documents = (
    pd.DataFrame({"topic": topics, "content": kmeans_documents})
    .groupby("topic", sort=True)["content"]
    .agg(" ".join)
)

label_vectorizer = TfidfVectorizer(tokenizer=tokenize_chinese_only, token_pattern=None, lowercase=False, min_df=1)
label_matrix = label_vectorizer.fit_transform(topic_documents)
label_terms = label_vectorizer.get_feature_names_out()


In [ ]:
# 生成每个主题的中文关键词列表。
# 注意：这里完全基于原始中文文本的 TF-IDF，不使用 Qwen embedding 或 BERTopic 原生词权。
topic_word_map = {}
topic_label_map = {}
for topic_id, row in zip(topic_documents.index, label_matrix.toarray()):
    weighted_row = row.copy()
    for index, term in enumerate(label_terms):
        if term in keyword_prefer_words:
            weighted_row[index] *= 1.35

    top_indices = [index for index in weighted_row.argsort()[::-1] if row[index] > 0][:10]
    words = [label_terms[index] for index in top_indices]
    topic_word_map[topic_id] = words
    topic_label_map[topic_id] = " / ".join(words[:5]) or "（无中文关键词）"


def topic_keywords(topic_id: int) -> str:
    return topic_label_map.get(topic_id, "（无中文关键词）")


In [ ]:
# 主题命名候选表：在后续 KMeans 汇总和画图前，先给人工/大模型判断主题标题。
def compact_counts(series: pd.Series, top_n: int = 3) -> str:
    counts = series.value_counts().head(top_n)
    return "；".join(f"{name}({count})" for name, count in counts.items())


def representative_text_for_topic(topic_id: int, top_n: int = 2) -> str:
    """直接从 result_df 选代表文本，不依赖 BERTopic 原生代表文档。"""
    topic_rows = kmeans_df.loc[kmeans_df["topic"].eq(topic_id)].copy()
    if topic_rows.empty:
        return ""

    if "互动总量" in topic_rows.columns:
        topic_rows = topic_rows.sort_values("互动总量", ascending=False)

    text_column = "source_content" if "source_content" in topic_rows.columns else "content"
    previews = [clean_preview(text) for text in topic_rows[text_column].dropna().head(top_n)]
    return "；".join(preview for preview in previews if preview)


representative_docs = {
    topic_id: representative_text_for_topic(topic_id)
    for topic_id in sorted(kmeans_df["topic"].unique())
}

topic_title_candidates = pd.DataFrame({"Topic": sorted(kmeans_df["topic"].unique())})
topic_title_candidates["关键词"] = topic_title_candidates["Topic"].map(topic_keywords)
topic_title_candidates["建议主题标题"] = topic_title_candidates["关键词"].str.replace(" / ", "、", regex=False)
topic_title_candidates["站点构成"] = topic_title_candidates["Topic"].map(
    lambda topic_id: compact_counts(kmeans_df.loc[kmeans_df["topic"].eq(topic_id), "station"])
)
topic_title_candidates["代表文本"] = topic_title_candidates["Topic"].map(representative_docs)

display(topic_title_candidates)

topic_title_candidate_path = RESULT_PATH.with_name("topic_title_candidates.csv")
topic_title_candidates.to_csv(topic_title_candidate_path, index=False, encoding="utf-8-sig")
print(f"主题标题候选表已保存到：{topic_title_candidate_path.resolve()}")

# 默认主题标题先使用建议标题；下一个单元可以人工覆盖这个字典。
topic_titles = dict(zip(topic_title_candidates["Topic"], topic_title_candidates["建议主题标题"]))


def topic_title(topic_id: int) -> str:
    return topic_titles.get(topic_id, topic_keywords(topic_id))

In [ ]:
# 在这里人工修改主题标题；也可以把上一个候选表交给大模型后，把结果粘回这个字典。
# 未填写或缺失的主题会自动回退到上一个单元生成的“建议主题标题/关键词”。

# 示例：
# topic_titles.update({
#     0: "接驳动线与通行效率问题",
#     1: "候车空间与商业服务体验",
# })


def topic_title(topic_id: int) -> str:
    return topic_titles.get(topic_id, topic_keywords(topic_id))

print(f"已加载 {len(topic_titles)} 个主题标题，可在本单元手动修改。")


### KMeans：3. 聚类质量评价

这一组指标用于检查当前 KMeans-BERTopic 的聚类结构与主题关键词解释性。


### KMeans：3.1 评价函数

定义聚类结构指标和主题解释性指标，再执行当前 KMeans-BERTopic 的评价。


In [ ]:
EVALUATION_PATH = RESULT_PATH.with_name("bertopic_evaluation_metrics.csv")


def _default_tokenizer(text: str) -> list[str]:
    return [token for token in re.findall(r"\w+", str(text).lower()) if len(token) > 1]


def _safe_cluster_metric(metric_func, embeddings_array, labels_array, **kwargs) -> float:
    """聚类指标的安全包装：主题数不足时返回 NaN。"""
    labels_array = np.asarray(labels_array)
    valid_mask = labels_array != -1
    x = np.asarray(embeddings_array)[valid_mask]
    y = labels_array[valid_mask]
    unique_labels = np.unique(y)
    if len(unique_labels) < 2 or len(y) <= len(unique_labels):
        return np.nan
    try:
        return float(metric_func(x, y, **kwargs))
    except ValueError:
        return np.nan


In [ ]:
def _topic_word_lists(topic_model: BERTopic, topic_ids, top_n_words: int = 10) -> list[list[str]]:
    """取主题关键词列表。优先使用中文 TF-IDF 关键词，避免 BERTopic 原生词表噪声。"""
    word_lists = []
    for topic_id in sorted(int(topic) for topic in topic_ids if int(topic) != -1):
        words = topic_word_map.get(topic_id, [])[:top_n_words]
        if words:
            word_lists.append(words)
    return word_lists


def topic_diversity_score(topic_word_lists: list[list[str]]) -> float:
    """主题多样性：不同主题 Top 词的去重比例。"""
    all_words = [word for words in topic_word_lists for word in words]
    if not all_words:
        return np.nan
    return len(set(all_words)) / len(all_words)


In [ ]:
def topic_coherence_npmi(
    docs: list[str],
    topic_word_lists: list[list[str]],
    tokenizer=None,
    eps: float = 1e-12,
) -> float:
    """基于 NPMI 的主题一致性，用关键词共现衡量主题解释性。"""
    tokenizer = tokenizer or _default_tokenizer
    tokenized_docs = [set(tokenizer(doc)) for doc in docs]
    n_docs = len(tokenized_docs)
    if n_docs == 0:
        return np.nan

    doc_freq = {}
    for tokens in tokenized_docs:
        for token in tokens:
            doc_freq[token] = doc_freq.get(token, 0) + 1

    pair_scores = []
    for words in topic_word_lists:
        words = [word for word in dict.fromkeys(words) if doc_freq.get(word, 0) > 0]
        if len(words) < 2:
            continue
        for i, word_i in enumerate(words[:-1]):
            docs_i = doc_freq[word_i]
            for word_j in words[i + 1:]:
                docs_j = doc_freq[word_j]
                co_docs = sum(1 for tokens in tokenized_docs if word_i in tokens and word_j in tokens)
                p_i = docs_i / n_docs
                p_j = docs_j / n_docs
                p_ij = (co_docs + eps) / n_docs
                npmi = np.log(p_ij / (p_i * p_j)) / (-np.log(p_ij))
                pair_scores.append(npmi)

    return float(np.mean(pair_scores)) if pair_scores else np.nan


In [ ]:
def evaluate_bertopic_model(
    topic_model: BERTopic,
    docs: list[str],
    embeddings_array,
    labels,
    experiment_name: str = "kmeans_bertopic",
    tokenizer=tokenize_zh,
    top_n_words: int = 10,
) -> pd.DataFrame:
    """汇总聚类结构指标和主题解释性指标。"""
    labels_array = np.asarray(labels)
    valid_labels = labels_array[labels_array != -1]
    topic_ids = np.unique(valid_labels)
    word_lists = _topic_word_lists(topic_model, topic_ids, top_n_words=top_n_words)

    metrics = {
        "experiment": experiment_name,
        "n_documents": int(len(docs)),
        "n_topics": int(len(topic_ids)),
        "silhouette_score": _safe_cluster_metric(silhouette_score, embeddings_array, labels_array, metric="cosine"),
        "davies_bouldin_index": _safe_cluster_metric(davies_bouldin_score, embeddings_array, labels_array),
        "calinski_harabasz_index": _safe_cluster_metric(calinski_harabasz_score, embeddings_array, labels_array),
        "topic_coherence_npmi": topic_coherence_npmi(docs, word_lists, tokenizer=tokenizer),
        "topic_diversity": topic_diversity_score(word_lists),
    }
    return pd.DataFrame([metrics])


In [ ]:
# 执行当前 KMeans-BERTopic 结果的五指标评价并保存。
bertopic_evaluation = evaluate_bertopic_model(
    topic_model,
    kmeans_documents,
    kmeans_embeddings,
    topics,
    experiment_name=f"kmeans_bertopic_k{N_TOPICS}",
    tokenizer=tokenize_zh,
    top_n_words=N_WORDS,
)
bertopic_evaluation.to_csv(EVALUATION_PATH, index=False, encoding="utf-8-sig")

display(bertopic_evaluation)
print(f"BERTopic 五指标评价已保存到：{EVALUATION_PATH.resolve()}")


### KMeans：4. 计算 PageRank 影响力

KMeans 影响力在每个 topic 内建立语义相似网络，再用 PageRank 衡量帖子在主题内部的中心性。


### KMeans：4.1 语义相似网络与 PageRank 影响力

这一组单元把 KMeans 聚类后的文本构造成主题内语义网络，并计算帖子级影响力。


In [ ]:
# 主题编号已在主题命名前写入 result_df；这里确认后续 KMeans 网络分析所需字段存在。
if "topic" not in kmeans_df.columns:
    kmeans_df["topic"] = topics


In [ ]:
# KMeans PageRank：只在同一 KMeans topic 内部连边。
kmeans_similarity_matrix = np.asarray(kmeans_embeddings) @ np.asarray(kmeans_embeddings).T
np.fill_diagonal(kmeans_similarity_matrix, 0)

kmeans_graph = build_group_semantic_graph(kmeans_df["topic"], similarity_matrix_input=kmeans_similarity_matrix)
print(f"KMeans 主题语义网络：{kmeans_graph.number_of_nodes():,} 个节点，{kmeans_graph.number_of_edges():,} 条边")


In [ ]:
kmeans_pagerank = personalized_pagerank(kmeans_graph, kmeans_df["互动总量"])

kmeans_df["kmeans_pagerank"] = kmeans_pagerank.to_numpy()
kmeans_df["kmeans_impact_score"] = (kmeans_df["kmeans_pagerank"] * 10000).round(4)
kmeans_df["kmeans_network_centrality"] = percentile_score(kmeans_df["kmeans_pagerank"]).round(2)
kmeans_df["topic_recurrence"] = percentile_score(kmeans_df["topic"].map(kmeans_df["topic"].value_counts())).round(2)
kmeans_df["topic_priority_layer"] = pareto_front_rank(kmeans_df, ["kmeans_network_centrality", "topic_recurrence", "互动强度"])

kmeans_df[["id", "topic", "kmeans_impact_score", "kmeans_network_centrality", "互动总量", "topic_recurrence", "topic_priority_layer"]].head()


### KMeans：4.2 主题层汇总与导出

这里把 KMeans PageRank、互动量和复现度聚合到主题层，生成主题摘要表。


In [ ]:
# 汇总每个 KMeans 主题的网络影响力、互动量和可读解释信息。
topic_summary = (
    kmeans_df.groupby("topic", as_index=False)
    .agg(Count=("id", "count"))
    .rename(columns={"topic": "Topic"})
)
topic_summary["Representative_Docs"] = topic_summary["Topic"].map(representative_docs)
topic_summary["关键词"] = topic_summary["Topic"].map(topic_keywords)
topic_summary["主题标题"] = topic_summary["Topic"].map(topic_title)

total_docs = len(kmeans_df)
total_kmeans_impact = kmeans_df["kmeans_impact_score"].sum()
total_engagement = kmeans_df["互动总量"].sum()

topic_metrics = kmeans_df.groupby("topic", as_index=False).agg(
    点赞数=("liked_count_num", "sum"),
    收藏数=("collected_count_num", "sum"),
    评论数=("comment_count_num", "sum"),
    转发数=("share_count_num", "sum"),
    互动总量=("互动总量", "sum"),
    平均互动强度=("互动强度", "mean"),
    综合影响分=("kmeans_impact_score", "sum"),
    平均网络中心性=("kmeans_network_centrality", "mean"),
    主题复现度=("topic_recurrence", "mean"),
)

topic_summary = topic_summary.merge(topic_metrics, left_on="Topic", right_on="topic", how="left").drop(columns="topic")
topic_summary["文本占比"] = topic_summary["Count"] / total_docs
topic_summary["影响占比"] = topic_summary["综合影响分"] / total_kmeans_impact if total_kmeans_impact else 0
topic_summary["互动占比"] = topic_summary["互动总量"] / total_engagement if total_engagement else 0
topic_summary["主题优先级层"] = pareto_front_rank(topic_summary, ["综合影响分", "主题复现度", "互动总量"])


In [ ]:
# 整理 KMeans 主题汇总表；不在这里混入 LLM 五类业务构成。
score_columns = ["综合影响分", "平均网络中心性", "平均互动强度", "主题复现度"]
topic_summary[score_columns] = topic_summary[score_columns].round(2)
topic_summary = topic_summary.sort_values(["主题优先级层", "综合影响分"], ascending=[True, False]).reset_index(drop=True)

impact_columns = [
    "主题优先级层", "Topic", "Count", "文本占比", "综合影响分", "影响占比", "互动总量", "互动占比",
    "平均网络中心性", "平均互动强度", "主题复现度", "主题标题", "关键词", "Representative_Docs",
]
display(topic_summary[impact_columns])


In [ ]:
# 导出 KMeans 节点级结果和主题级汇总结果。
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
kmeans_df.sort_values(["topic_priority_layer", "topic", "id"]).to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
topic_summary.to_csv(RESULT_PATH.with_name("kmeans_topic_summary.csv"), index=False, encoding="utf-8-sig")

print(f"KMeans 节点级结果已保存到：{RESULT_PATH.resolve()}")
print(f"KMeans 主题汇总已保存到：{RESULT_PATH.with_name('kmeans_topic_summary.csv').resolve()}")


### KMeans：5. 影响力与帖子可视化、关键词 barchart

这里包含 KMeans 主题规模图、帖子级影响力散点图、embedding 二维分布图、中文关键词 barchart 和主题层级图。


In [ ]:
# KMeans 主题规模条形图。
chart_data = topic_summary.copy()
chart_data["主题"] = chart_data.apply(
    lambda row: f"主题 {row['Topic']}：{topic_title(row['Topic'])}", axis=1
)
fig_topic_count = px.bar(
    chart_data.sort_values("Count"),
    x="Count",
    y="主题",
    orientation="h",
    text="Count",
    title="KMeans 主题分布",
    labels={"Count": "文本数", "主题": "主题标题"},
)
fig_topic_count.update_traces(textposition="outside")
fig_topic_count.update_layout(height=max(450, 55 * len(chart_data)), margin={"l": 260, "r": 40, "t": 60, "b": 40})
fig_topic_count.show()

topic_count_chart_path = RESULT_PATH.with_name("kmeans_topic_count_distribution.html")
fig_topic_count.write_html(topic_count_chart_path, include_plotlyjs=True)
print(f"KMeans 主题规模图已保存到：{topic_count_chart_path.resolve()}")


In [ ]:
# KMeans embedding 二维散点图：用 PCA 仅做展示，不参与聚类。
pca = PCA(n_components=2, random_state=42)
embedding_2d = pca.fit_transform(kmeans_embeddings)

scatter_df = kmeans_df.copy()
scatter_df["pca_x"] = embedding_2d[:, 0]
scatter_df["pca_y"] = embedding_2d[:, 1]
scatter_df["主题"] = scatter_df["topic"].map(lambda topic_id: f"主题 {topic_id}：{topic_title(topic_id)}")
scatter_df["互动气泡"] = np.log1p(scatter_df["互动总量"].astype(float)) + 4
scatter_df["文本预览"] = scatter_df["source_content"].fillna(scatter_df["content"]).map(lambda text: clean_preview(text, max_chars=80))

fig_kmeans_scatter = px.scatter(
    scatter_df,
    x="pca_x",
    y="pca_y",
    color="主题",
    size="互动气泡",
    hover_data={
        "station": True,
        "主题": True,
        "互动总量": True,
        "文本预览": True,
        "pca_x": False,
        "pca_y": False,
        "互动气泡": False,
    },
    title="KMeans 主题二维分布（PCA 展示）",
    labels={"station": "站点"},
)
fig_kmeans_scatter.update_traces(marker={"opacity": 0.78, "line": {"width": 0.4, "color": "white"}})
fig_kmeans_scatter.update_layout(height=620, margin={"l": 40, "r": 260, "t": 70, "b": 40}, legend_title_text="主题")
fig_kmeans_scatter.show()

kmeans_scatter_path = RESULT_PATH.with_name("kmeans_topic_embedding_scatter.html")
fig_kmeans_scatter.write_html(kmeans_scatter_path, include_plotlyjs=True)
print(f"KMeans 主题二维散点图已保存到：{kmeans_scatter_path.resolve()}")


In [ ]:
# KMeans 帖子级影响力图：每个点是一条帖子。
kmeans_post_df = kmeans_df.copy()
kmeans_post_df["主题"] = kmeans_post_df["topic"].map(lambda topic_id: f"主题 {topic_id}：{topic_title(topic_id)}")
kmeans_post_df["文本预览"] = kmeans_post_df["source_content"].fillna(kmeans_post_df["content"]).map(lambda text: clean_preview(text, max_chars=90))

fig_kmeans_posts = px.scatter(
    kmeans_post_df,
    x="kmeans_impact_score",
    y="互动总量",
    color="主题",
    size="互动强度",
    hover_data={
        "station": True,
        "主题": True,
        "kmeans_impact_score": ":.2f",
        "kmeans_network_centrality": ":.1f",
        "topic_recurrence": ":.1f",
        "文本预览": True,
        "互动强度": False,
    },
    title="KMeans：帖子影响力与互动量",
    labels={"kmeans_impact_score": "KMeans topic 内 PageRank 影响分"},
)
fig_kmeans_posts.update_traces(marker={"opacity": 0.75, "line": {"width": 0.4, "color": "white"}})
fig_kmeans_posts.update_layout(height=620, margin={"l": 60, "r": 260, "t": 70, "b": 50}, legend_title_text="KMeans 主题")
fig_kmeans_posts.show()

kmeans_posts_path = RESULT_PATH.with_name("kmeans_topic_post_influence.html")
fig_kmeans_posts.write_html(kmeans_posts_path, include_plotlyjs=True)

kmeans_top_posts = kmeans_post_df.sort_values("kmeans_impact_score", ascending=False).head(30)
kmeans_top_posts[["id", "station", "主题", "kmeans_impact_score", "kmeans_network_centrality", "互动总量", "topic_recurrence", "文本预览"]].to_csv(
    RESULT_PATH.with_name("kmeans_top_influence_posts.csv"), index=False, encoding="utf-8-sig"
)
display(kmeans_top_posts[["id", "station", "主题", "kmeans_impact_score", "kmeans_network_centrality", "互动总量", "topic_recurrence", "文本预览"]])
print(f"KMeans 帖子影响力图已保存到：{kmeans_posts_path.resolve()}")


In [ ]:
# 中文关键词 barchart：只使用原始中文文本 TF-IDF，不使用 BERTopic 原生词图。
N_WORDS = 8
topic_ids = list(topic_documents.index)
n_cols = 4
n_rows = (len(topic_ids) + n_cols - 1) // n_cols

# 图表标题在人工主题命名后创建，确保使用下方确认的主题名称。
colors = ["#D55E00", "#0072B2", "#CC79A7", "#E69F00", "#56B4E9", "#009E73", "#F0E442"]


In [ ]:
def add_topic_keyword_bar(fig, index: int, topic_id: int, row) -> None:
    """向 subplot 添加一个主题的 Top 中文关键词条形图。"""
    weighted_row = row.copy()
    for item, term in enumerate(label_terms):
        if term in keyword_prefer_words:
            weighted_row[item] *= 1.35

    top_indices = [item for item in weighted_row.argsort()[::-1] if row[item] > 0][:N_WORDS]
    words = [label_terms[item] for item in top_indices][::-1]
    weights = [weighted_row[item] for item in top_indices][::-1]

    fig.add_trace(
        go.Bar(x=weights, y=words, orientation="h", marker_color=colors[index % len(colors)], showlegend=False),
        row=index // n_cols + 1,
        col=index % n_cols + 1,
    )


In [ ]:
### LLM Topic Name

# 基于当前主题关键词、代表帖子和图文综合描述确认的主题名称。
topic_titles.update({
    0: "候车环境与进站体验",
    1: "丰台站出站换乘动线",
    2: "机场运行延误与旅客滞留",
    3: "首都机场航站楼服务与接驳体验",
    4: "网约车接驳与运力不足",
    5: "列车晚点与候车滞留",
    6: "通州站空间布局与通行体验",
    7: "机场线票价与接驳成本",
    8: "站台吸烟与空气环境",
    9: "目的车站混淆与改签困扰",
})
if "topic_summary" in globals():
    topic_summary["主题标题"] = topic_summary["Topic"].map(topic_title)
    topic_summary.to_csv(RESULT_PATH.with_name("kmeans_topic_summary.csv"), index=False, encoding="utf-8-sig")
print(f"已更新 {len(topic_titles)} 个主题标题。")


In [ ]:
titles = [f"主题 {topic_id}：{topic_title(topic_id)}" for topic_id in topic_ids]
fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=titles)

for index, (topic_id, row) in enumerate(zip(topic_ids, label_matrix.toarray())):
    add_topic_keyword_bar(fig, index, topic_id, row)

fig.update_layout(
    title="中文主题关键词权重（TF-IDF）",
    height=max(450, 330 * n_rows),
    margin={"l": 80, "r": 30, "t": 100, "b": 40},
)
fig.update_xaxes(title_text="TF-IDF 权重")
fig.show()


In [ ]:
chart_path = RESULT_PATH.with_name("bertopic_chinese_keywords_barchart.html")
fig.write_html(chart_path, include_plotlyjs=True)
print(f"交互式图表已保存到：{chart_path.resolve()}")


In [ ]:
# KMeans 主题级文本占比与影响占比对比图。
# 仅使用 KMeans 主题汇总结果，比较主题规模与主题影响力。
topic_chart_data = topic_summary.copy()
topic_chart_data["主题"] = topic_chart_data.apply(
    lambda row: f"主题 {int(row['Topic'])}：{topic_title(row['Topic'])}", axis=1
)
topic_chart_data = topic_chart_data.sort_values(
    ["主题优先级层", "综合影响分"], ascending=[True, False]
).reset_index(drop=True)
topic_order = topic_chart_data["主题"].tolist()

topic_distribution_chart_data = topic_chart_data.melt(
    id_vars="主题",
    value_vars=["文本占比", "影响占比"],
    var_name="指标",
    value_name="占比",
)

fig_kmeans_topic_distribution = px.bar(
    topic_distribution_chart_data,
    x="占比",
    y="主题",
    color="指标",
    orientation="h",
    barmode="group",
    text=topic_distribution_chart_data["占比"].map(lambda value: f"{value:.1%}"),
    title="KMeans 主题：文本占比 vs 影响占比",
    labels={"主题": "KMeans 主题", "占比": "占比"},
)
fig_kmeans_topic_distribution.update_traces(textposition="outside")
fig_kmeans_topic_distribution.update_layout(
    height=max(520, 62 * len(topic_chart_data)),
    margin={"l": 310, "r": 50, "t": 60, "b": 40},
)
fig_kmeans_topic_distribution.update_xaxes(tickformat=".0%")
fig_kmeans_topic_distribution.update_yaxes(categoryorder="array", categoryarray=topic_order[::-1])
fig_kmeans_topic_distribution.show()

kmeans_topic_distribution_chart_path = RESULT_PATH.with_name("kmeans_topic_distribution_vs_impact.html")
fig_kmeans_topic_distribution.write_html(kmeans_topic_distribution_chart_path, include_plotlyjs=True)
print(f"KMeans 主题占比图已保存到：{kmeans_topic_distribution_chart_path.resolve()}")


### 论文第 4.5 节补充图表

以下图表分别呈现代表帖的可回溯证据、主题在站场间的标准化分布，以及主题规模与综合影响力的关系。

In [ ]:
# 表 3：代表帖的原始文本、综合描述与主题归属。
# 选取三个高影响力主题中影响分最高的帖子；该表用于展示主题解释的可回溯证据。
CASE_TOPIC_IDS = [0, 1, 2]
CASE_TEXT_LIMIT = 90

def compact_table_text(value, limit=CASE_TEXT_LIMIT):
    text = str(value or "").replace("\n", " ").strip()
    if len(text) > limit:
        text = text[:limit].rstrip() + "……"
    return text

case_rows = (
    kmeans_df.loc[kmeans_df["topic"].isin(CASE_TOPIC_IDS)]
    .sort_values(["topic", "kmeans_impact_score"], ascending=[True, False])
    .groupby("topic", as_index=False)
    .head(1)
    .copy()
)
case_rows["主题"] = case_rows["topic"].map(lambda topic_id: f"主题 {topic_id}：{topic_title(topic_id)}")
case_rows["站场"] = case_rows["station"].map({
    "xizhan": "北京西站", "nanzhan": "北京南站", "beizhan": "北京北站",
    "fengtai": "北京丰台站", "chaoyang": "北京朝阳站", "qinghe": "北京清河站",
    "tongzhou": "北京通州站", "beijing": "北京站",
    "daxingjichang": "北京大兴国际机场", "shoudujichang": "北京首都国际机场",
}).fillna(case_rows["station"])
case_rows["原始短文本（节选）"] = case_rows["source_content"].map(compact_table_text)
case_rows["综合描述（节选）"] = case_rows["description"].fillna(case_rows["content"]).map(compact_table_text)
case_export_columns = ["id", "主题", "站场", "原始短文本（节选）", "综合描述（节选）", "kmeans_impact_score"]
case_rows[case_export_columns].to_csv(RESULT_PATH.with_name("multimodal_topic_cases.csv"), index=False, encoding="utf-8-sig")
case_table = case_rows[["主题", "站场", "原始短文本（节选）", "综合描述（节选）"]].reset_index(drop=True)
display(case_table)
print(f"代表帖对照表已保存到：{RESULT_PATH.with_name('multimodal_topic_cases.csv').resolve()}")


In [ ]:
# 图 4-3：负向舆情主题在各站场的标准化分布。
# 按站场内主题占比标准化，以避免样本量较大的站场主导比较。
STATION_ORDER = [
    "xizhan", "nanzhan", "beizhan", "fengtai", "chaoyang", "qinghe", "tongzhou", "beijing",
    "daxingjichang", "shoudujichang",
]
STATION_LABELS = {
    "xizhan": "北京西站", "nanzhan": "北京南站", "beizhan": "北京北站",
    "fengtai": "北京丰台站", "chaoyang": "北京朝阳站", "qinghe": "北京清河站",
    "tongzhou": "北京通州站", "beijing": "北京站",
    "daxingjichang": "北京大兴国际机场", "shoudujichang": "北京首都国际机场",
}
station_order = [station for station in STATION_ORDER if station in set(kmeans_df["station"])]
topic_order = topic_summary.sort_values("综合影响分", ascending=False)["Topic"].astype(int).tolist()
station_topic_count = pd.crosstab(kmeans_df["topic"], kmeans_df["station"]).reindex(index=topic_order, columns=station_order, fill_value=0)
station_total = kmeans_df["station"].value_counts().reindex(station_order, fill_value=0)
station_topic_share = station_topic_count.div(station_total.replace(0, np.nan), axis=1).fillna(0)
station_topic_share.to_csv(RESULT_PATH.with_name("kmeans_topic_station_share.csv"), encoding="utf-8-sig")

heatmap_x = [f"{STATION_LABELS.get(station, station)}<br>n={int(station_total[station])}" for station in station_order]
heatmap_y = [f"主题 {topic_id}：{topic_title(topic_id)}" for topic_id in topic_order]
heatmap_text = np.array([[f"{value:.1%}" for value in row] for row in station_topic_share.to_numpy()])

fig_topic_station_heatmap = go.Figure(data=go.Heatmap(
    z=station_topic_share.to_numpy(), x=heatmap_x, y=heatmap_y, text=heatmap_text, texttemplate="%{text}",
    colorscale=[[0, "#F7FBFF"], [0.5, "#9ECAE1"], [1, "#08519C"]],
    colorbar=dict(title="站场内<br>主题占比", tickformat=".0%"),
    hovertemplate="%{y}<br>%{x}<br>站场内主题占比：%{z:.1%}<extra></extra>",
))
fig_topic_station_heatmap.update_layout(
    title="图 4-3　负向舆情主题在各站场的标准化分布",
    width=1380, height=720, margin=dict(l=300, r=120, t=80, b=120),
)
fig_topic_station_heatmap.add_shape(type="line", x0=0.8, x1=0.8, y0=0, y1=1, xref="paper", yref="paper", line=dict(dash="dash", color="#7F8C8D", width=1))
fig_topic_station_heatmap.add_annotation(x=0.4, y=1.08, xref="paper", yref="paper", text="铁路客运站", showarrow=False)
fig_topic_station_heatmap.add_annotation(x=0.9, y=1.08, xref="paper", yref="paper", text="机场", showarrow=False)
fig_topic_station_heatmap.show()

topic_station_heatmap_path = RESULT_PATH.with_name("kmeans_topic_station_heatmap.html")
fig_topic_station_heatmap.write_html(topic_station_heatmap_path, include_plotlyjs=True)
try:
    fig_topic_station_heatmap.write_image(RESULT_PATH.with_name("kmeans_topic_station_heatmap.png"), width=2760, height=1440, scale=1)
except Exception as error:
    print(f"静态 PNG 未导出，请安装 kaleido 后重试：{error}")
print(f"主题×站场热力图已保存到：{topic_station_heatmap_path.resolve()}")


In [ ]:
# 图 4-4：主题规模与综合影响力四象限。
# 横轴为主题文本占比，纵轴为综合影响分占比，气泡大小表示互动总量。
topic_quadrant_data = topic_summary.copy()
topic_quadrant_data["主题"] = topic_quadrant_data["Topic"].map(lambda topic_id: f"主题 {int(topic_id)}：{topic_title(topic_id)}")
topic_quadrant_data["标签"] = topic_quadrant_data["Topic"].map(lambda topic_id: f"主题 {int(topic_id)}")
topic_quadrant_data["主题样本数"] = topic_quadrant_data["Count"].astype(int)
topic_count_reference = 1 / len(topic_quadrant_data)
impact_reference = 1 / len(topic_quadrant_data)

fig_topic_impact_quadrant = px.scatter(
    topic_quadrant_data, x="文本占比", y="影响占比", size="互动总量", text="标签",
    hover_name="主题",
    hover_data={
        "主题样本数": True, "综合影响分": ":.2f", "互动总量": ":.0f",
        "平均网络中心性": ":.2f", "文本占比": ":.1%", "影响占比": ":.1%", "标签": False,
    },
    size_max=54, color_discrete_sequence=["#2F6B8E"],
    title="图 4-4　主题规模与综合影响力四象限",
    labels={"文本占比": "文本占比", "影响占比": "综合影响分占比"},
)
fig_topic_impact_quadrant.update_traces(textposition="top center", marker=dict(line=dict(width=1, color="white"), opacity=0.82))
fig_topic_impact_quadrant.add_vline(x=topic_count_reference, line_dash="dash", line_color="#7F8C8D", line_width=1)
fig_topic_impact_quadrant.add_hline(y=impact_reference, line_dash="dash", line_color="#7F8C8D", line_width=1)
fig_topic_impact_quadrant.add_annotation(x=0.88, y=0.92, xref="paper", yref="paper", text="规模与影响均较高", showarrow=False)
fig_topic_impact_quadrant.add_annotation(x=0.12, y=0.92, xref="paper", yref="paper", text="影响相对较高", showarrow=False)
fig_topic_impact_quadrant.add_annotation(x=0.88, y=0.08, xref="paper", yref="paper", text="规模相对较高", showarrow=False)
fig_topic_impact_quadrant.add_annotation(x=0.12, y=0.08, xref="paper", yref="paper", text="规模与影响相对较低", showarrow=False)
fig_topic_impact_quadrant.update_layout(width=1120, height=720, margin=dict(l=90, r=60, t=80, b=70), showlegend=False)
fig_topic_impact_quadrant.update_xaxes(tickformat=".0%", zeroline=False)
fig_topic_impact_quadrant.update_yaxes(tickformat=".0%", zeroline=False)
fig_topic_impact_quadrant.show()

topic_impact_quadrant_path = RESULT_PATH.with_name("kmeans_topic_scale_impact_quadrant.html")
fig_topic_impact_quadrant.write_html(topic_impact_quadrant_path, include_plotlyjs=True)
try:
    fig_topic_impact_quadrant.write_image(RESULT_PATH.with_name("kmeans_topic_scale_impact_quadrant.png"), width=2240, height=1440, scale=1)
except Exception as error:
    print(f"静态 PNG 未导出，请安装 kaleido 后重试：{error}")
print(f"主题规模—影响力四象限图已保存到：{topic_impact_quadrant_path.resolve()}")


In [ ]:
# 主题层级结构图：用每个 topic 的平均 embedding 表示主题，再做层级聚类。
topic_ids = sorted(kmeans_df["topic"].unique())
topic_centroids = np.vstack([
    kmeans_embeddings[kmeans_df["topic"].to_numpy() == topic_id].mean(axis=0)
    for topic_id in topic_ids
])
topic_labels = [f"主题 {topic_id}：{topic_title(topic_id)}" for topic_id in topic_ids]

fig = ff.create_dendrogram(
    topic_centroids,
    labels=topic_labels,
    orientation="left",
    linkagefun=lambda values: linkage(values, method="ward"),
)
fig.update_layout(
    title="主题层级结构图（基于 topic embedding 均值）",
    width=1100,
    height=max(520, 70 * len(topic_ids)),
    margin={"l": 320, "r": 40, "t": 70, "b": 40},
    xaxis_title="主题距离",
)
fig.show()

hierarchy_path = RESULT_PATH.with_name("bertopic_topic_hierarchy.html")
fig.write_html(hierarchy_path, include_plotlyjs=True)
print(f"主题层级结构图已保存到：{hierarchy_path.resolve()}")


### KMeans 与业务类别交叉对照


In [ ]:
topic_business_crosstab = pd.crosstab(
    kmeans_df["topic"].map(lambda topic_id: f"主题 {topic_id}：{topic_title(topic_id)}"),
    kmeans_df["business_category_v2"].fillna("未分类"),
    normalize="index",
).round(3)

topic_business_crosstab.to_csv(
    RESULT_PATH.with_name("negative_topic_vs_business_category_crosstab.csv"),
    encoding="utf-8-sig",
)
display(topic_business_crosstab)
print(
    "负向主题与业务类别交叉表已保存到："
    f"{RESULT_PATH.with_name('negative_topic_vs_business_category_crosstab.csv').resolve()}"
)
